In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
#!/usr/bin/env python3
"""
Combined Dataset Generalization
================================
Splits all 3 datasets into train/val/test, combines them, and evaluates generalization.

Strategy:
- Dataset 1 (NCT-CRC-100K): Split 70% train, 15% val, 15% test
- Dataset 2 (CRC-VAL-7K): Split 70% train, 15% val, 15% test  
- Dataset 3 (LC25000): Split 70% train, 15% val, 15% test
- Train on: Combined train splits from all 3 datasets
- Validate on: Combined val splits from all 3 datasets
- Test on: Each dataset's test split separately + combined test
"""

import os
import json
import logging
import random
from pathlib import Path
from typing import Dict, List, Tuple
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR
import torchvision.transforms as transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, balanced_accuracy_score,
    roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


@dataclass
class Config:
    """Configuration."""
    dataset1_dir: str = "/kaggle/input/datasets/imrankhan77/nct-crc-he-100k/NCT-CRC-HE-100K"
    dataset2_dir: str = "/kaggle/input/datasets/imrankhan77/crc-val-he-7k/CRC-VAL-HE-7K"
    # LC25000 dataset from https://www.kaggle.com/datasets/antrnduy/lc25000
    dataset3_dir: str = "/kaggle/input/lc25000/lung_colon_image_set/colon_image_sets"
    output_dir: str = "/kaggle/working/combined_generalization_outputs"
    
    # Split ratios
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    
    # Training settings
    img_size: int = 224
    batch_size: int = 64
    num_workers: int = 4
    epochs: int = 10
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    use_amp: bool = True
    seed: int = 42
    
    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'models'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'figures'), exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_train_transforms(img_size: int = 224):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(90),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def get_val_transforms(img_size: int = 224):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


class ImageDataset(Dataset):
    """Dataset with binary classification: Normal vs Tumor."""
    
    # Mapping for NCT-CRC and CRC-VAL datasets (9 classes → 2 classes)
    NCT_TO_BINARY = {
        'ADI': 0,    # Adipose → Normal
        'BACK': 0,   # Background → Normal
        'DEB': 0,    # Debris → Normal
        'LYM': 0,    # Lymphocytes → Normal
        'MUC': 0,    # Mucus → Normal
        'MUS': 0,    # Smooth muscle → Normal
        'NORM': 0,   # Normal colon mucosa → Normal
        'STR': 1,    # Cancer-associated stroma → Tumor
        'TUM': 1,    # Colorectal adenocarcinoma → Tumor
    }
    
    # Mapping for LC25000 dataset (2 classes → 2 classes)
    LC_TO_BINARY = {
        'colon_n': 0,    # Colon benign → Normal
        'colon_aca': 1,  # Colon adenocarcinoma → Tumor
    }
    
    def __init__(self, root_dir: str, transform=None, source_name: str = "Unknown"):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.source_name = source_name
        self.samples = []
        self.labels = []
        self.class_names = ['Normal', 'Tumor']  # Binary classification
        self._load_dataset()
    
    def _load_dataset(self):
        logger.info(f"Loading {self.source_name} from {self.root_dir}")
        class_dirs = sorted([d for d in self.root_dir.iterdir() if d.is_dir()])
        original_classes = [d.name for d in class_dirs]
        
        # Detect dataset type
        is_lc25000 = any(cls in self.LC_TO_BINARY for cls in original_classes)
        mapping = self.LC_TO_BINARY if is_lc25000 else self.NCT_TO_BINARY
        
        logger.info(f"  Original classes: {original_classes}")
        logger.info(f"  Mapping to binary: Normal (0) vs Tumor (1)")
        
        normal_count = 0
        tumor_count = 0
        
        for class_dir in class_dirs:
            class_name = class_dir.name
            
            # Skip if class not in mapping
            if class_name not in mapping:
                logger.warning(f"  ⚠ Skipping unknown class: {class_name}")
                continue
            
            binary_label = mapping[class_name]
            
            image_files = list(class_dir.glob('*'))
            for img_path in image_files:
                if img_path.suffix.lower() in ['.tif', '.png', '.jpg', '.jpeg']:
                    self.samples.append(str(img_path))
                    self.labels.append(binary_label)
                    
                    if binary_label == 0:
                        normal_count += 1
                    else:
                        tumor_count += 1
        
        logger.info(f"  Loaded {len(self.samples)} images")
        logger.info(f"    Normal: {normal_count} images")
        logger.info(f"    Tumor:  {tumor_count} images")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            logger.error(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224))
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


class ColonCancerClassifier(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        weights = EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        self.backbone = efficientnet_v2_s(weights=weights)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        if features.dim() > 2:
            features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        return self.classifier(features)


class Trainer:
    def __init__(self, model: nn.Module, train_loader: DataLoader, 
                 val_loader: DataLoader, config: Config):
        self.model = model.to(config.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = config.device
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        self.scheduler = OneCycleLR(self.optimizer, max_lr=config.learning_rate,
                                    epochs=config.epochs, steps_per_epoch=len(train_loader))
        self.scaler = GradScaler(enabled=config.use_amp)
        self.best_val_acc = 0.0
    
    def train_epoch(self, epoch: int):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config.epochs} [Train]")
        for images, labels in pbar:
            images, labels = images.to(self.device), labels.to(self.device)
            
            self.optimizer.zero_grad()
            with autocast(enabled=self.config.use_amp):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
            
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
        
        return running_loss / len(self.train_loader.dataset), correct / total
    
    @torch.no_grad()
    def validate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in tqdm(self.val_loader, desc="[Validation]"):
            images, labels = images.to(self.device), labels.to(self.device)
            
            with autocast(enabled=self.config.use_amp):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        return running_loss / len(self.val_loader.dataset), correct / total
    
    def train(self):
        logger.info("\n" + "=" * 80)
        logger.info("TRAINING ON COMBINED DATASETS")
        logger.info("=" * 80)
        
        for epoch in range(self.config.epochs):
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate()
            
            logger.info(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
                       f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
            
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.save_checkpoint(epoch)
                logger.info(f"  ✓ Best model saved (Val Acc: {val_acc*100:.2f}%)")
        
        logger.info(f"\n✓ Training complete! Best Val Acc: {self.best_val_acc*100:.2f}%")
    
    def save_checkpoint(self, epoch: int):
        path = os.path.join(self.config.output_dir, 'models', 'best_model.pth')
        torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                   'best_val_acc': self.best_val_acc}, path)
    
    def load_best_model(self):
        path = os.path.join(self.config.output_dir, 'models', 'best_model.pth')
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        logger.info(f"✓ Loaded best model (Val Acc: {checkpoint['best_val_acc']*100:.2f}%)")


class Evaluator:
    def __init__(self, model: nn.Module, config: Config):
        self.model = model
        self.config = config
        self.device = config.device
        self.results = {}
    
    @torch.no_grad()
    def evaluate_dataset(self, dataloader: DataLoader, dataset_name: str, class_names: List[str]) -> Dict:
        logger.info(f"\n{'=' * 80}")
        logger.info(f"TESTING: {dataset_name}")
        logger.info(f"{'=' * 80}")
        
        self.model.eval()
        all_preds = []
        all_labels = []
        all_probs = []
        
        for images, labels in tqdm(dataloader, desc=f"Processing {dataset_name}"):
            images = images.to(self.device)
            outputs = self.model(images)
            probs = F.softmax(outputs, dim=1)
            preds = outputs.argmax(1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.append(probs.cpu().numpy())
        
        preds_np = np.array(all_preds)
        labels_np = np.array(all_labels)
        probs_np = np.vstack(all_probs)
        
        # Metrics
        accuracy = accuracy_score(labels_np, preds_np)
        balanced_acc = balanced_accuracy_score(labels_np, preds_np)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_np, preds_np, average='weighted', zero_division=0
        )
        
        # ROC-AUC
        try:
            n_classes = len(class_names)
            labels_bin = label_binarize(labels_np, classes=range(n_classes))
            if n_classes == 2:
                roc_auc = roc_auc_score(labels_np, probs_np[:, 1])
            else:
                roc_auc = roc_auc_score(labels_bin, probs_np, average='weighted', multi_class='ovr')
        except:
            roc_auc = 0.0
        
        results = {
            'dataset': dataset_name,
            'num_samples': len(labels_np),
            'num_classes': len(class_names),
            'class_names': class_names,
            'accuracy': accuracy,
            'balanced_accuracy': balanced_acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'roc_auc': roc_auc,
            'confusion_matrix': confusion_matrix(labels_np, preds_np),
            'predictions': preds_np,
            'labels': labels_np,
            'probabilities': probs_np,
        }
        
        logger.info(f"\n📊 RESULTS:")
        logger.info(f"  Samples:           {len(labels_np)}")
        logger.info(f"  Accuracy:          {accuracy*100:.2f}%")
        logger.info(f"  Balanced Accuracy: {balanced_acc*100:.2f}%")
        logger.info(f"  Precision:         {precision:.4f}")
        logger.info(f"  Recall:            {recall:.4f}")
        logger.info(f"  F1-Score:          {f1:.4f}")
        logger.info(f"  ROC-AUC:           {roc_auc:.4f}")
        
        self.results[dataset_name] = results
        return results
    
    def plot_confusion_matrices(self):
        n_datasets = len(self.results)
        if n_datasets == 0:
            return
        
        fig, axes = plt.subplots(1, n_datasets, figsize=(8*n_datasets, 6))
        if n_datasets == 1:
            axes = [axes]
        
        for idx, (dataset_name, results) in enumerate(self.results.items()):
            cm = results['confusion_matrix']
            class_names = results['class_names']
            cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)
            
            sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                       xticklabels=class_names, yticklabels=class_names,
                       ax=axes[idx], cbar_kws={'label': 'Proportion'})
            axes[idx].set_title(f'Confusion Matrix\n{dataset_name}', fontsize=12, fontweight='bold')
            axes[idx].set_ylabel('True Label')
            axes[idx].set_xlabel('Predicted Label')
            axes[idx].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        output_path = os.path.join(self.config.output_dir, 'figures', 'confusion_matrices.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"✓ Confusion matrices saved: {output_path}")
        plt.close()
    
    def plot_roc_curves(self):
        n_datasets = len(self.results)
        if n_datasets == 0:
            return
        
        fig, axes = plt.subplots(1, n_datasets, figsize=(8*n_datasets, 6))
        if n_datasets == 1:
            axes = [axes]
        
        for idx, (dataset_name, results) in enumerate(self.results.items()):
            labels = results['labels']
            probs = results['probabilities']
            class_names = results['class_names']
            n_classes = len(class_names)
            
            labels_bin = label_binarize(labels, classes=range(n_classes))
            
            if n_classes == 2:
                fpr, tpr, _ = roc_curve(labels, probs[:, 1])
                roc_auc_val = auc(fpr, tpr)
                axes[idx].plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc_val:.3f})')
            else:
                for i in range(min(n_classes, 5)):  # Plot max 5 classes
                    fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
                    roc_auc_val = auc(fpr, tpr)
                    axes[idx].plot(fpr, tpr, lw=2, label=f'{class_names[i]} (AUC={roc_auc_val:.2f})')
            
            axes[idx].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
            axes[idx].set_xlim([0.0, 1.0])
            axes[idx].set_ylim([0.0, 1.05])
            axes[idx].set_xlabel('False Positive Rate')
            axes[idx].set_ylabel('True Positive Rate')
            axes[idx].set_title(f'ROC Curves\n{dataset_name}', fontsize=12, fontweight='bold')
            axes[idx].legend(loc='lower right', fontsize=8)
            axes[idx].grid(alpha=0.3)
        
        plt.tight_layout()
        output_path = os.path.join(self.config.output_dir, 'figures', 'roc_curves.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"✓ ROC curves saved: {output_path}")
        plt.close()
    
    def compare_results(self):
        logger.info("\n" + "=" * 80)
        logger.info("GENERALIZATION COMPARISON ACROSS DATASETS")
        logger.info("=" * 80)
        
        data = []
        for name, res in self.results.items():
            data.append({
                'Dataset': name,
                'Samples': res['num_samples'],
                'Accuracy': f"{res['accuracy']*100:.2f}%",
                'Balanced_Acc': f"{res['balanced_accuracy']*100:.2f}%",
                'F1-Score': f"{res['f1_score']:.4f}",
                'ROC-AUC': f"{res['roc_auc']:.4f}",
            })
        
        df = pd.DataFrame(data)
        logger.info(f"\n{df.to_string(index=False)}")
        
        if len(self.results) > 1:
            accuracies = [r['accuracy'] for r in self.results.values()]
            gap = (max(accuracies) - min(accuracies)) * 100
            
            logger.info("\n" + "=" * 80)
            logger.info("GENERALIZATION GAP ANALYSIS")
            logger.info("=" * 80)
            logger.info(f"  Best:  {max(accuracies)*100:.2f}%")
            logger.info(f"  Worst: {min(accuracies)*100:.2f}%")
            logger.info(f"  Gap:   {gap:.2f}%")
            logger.info(f"  Mean:  {np.mean(accuracies)*100:.2f}%")
            logger.info(f"  Std:   {np.std(accuracies)*100:.2f}%")
            
            logger.info(f"\n💡 GENERALIZATION QUALITY:")
            if gap < 5:
                logger.info("  ✓ EXCELLENT - Model generalizes very well across datasets (gap < 5%)")
            elif gap < 10:
                logger.info("  ✓ GOOD - Model generalizes well (gap < 10%)")
            elif gap < 15:
                logger.info("  ⚠ MODERATE - Some variation across datasets (gap < 15%)")
            else:
                logger.info("  ✗ POOR - Significant performance variation (gap >= 15%)")


def plot_class_distribution(results: Dict, config: Config):
    """Plot class distribution for each test dataset."""
    n_datasets = len(results)
    if n_datasets == 0:
        return
    
    fig, axes = plt.subplots(1, n_datasets, figsize=(6*n_datasets, 5))
    if n_datasets == 1:
        axes = [axes]
    
    for idx, (dataset_name, res) in enumerate(results.items()):
        labels = res['labels']
        class_names = res['class_names']
        
        # Count samples per class
        unique, counts = np.unique(labels, return_counts=True)
        
        # Create bar plot
        bars = axes[idx].bar(range(len(unique)), counts, color=['skyblue', 'coral'], 
                            edgecolor='black', alpha=0.8)
        axes[idx].set_xticks(range(len(unique)))
        axes[idx].set_xticklabels([class_names[i] for i in unique])
        axes[idx].set_ylabel('Number of Samples', fontsize=11)
        axes[idx].set_title(f'Class Distribution\n{dataset_name}', fontsize=12, fontweight='bold')
        axes[idx].grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        for i, (bar, count) in enumerate(zip(bars, counts)):
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                          f'{count}\n({count/sum(counts)*100:.1f}%)',
                          ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    output_path = os.path.join(config.output_dir, 'figures', 'class_distribution.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    logger.info(f"✓ Class distribution saved: {output_path}")
    plt.close()


def main():
    config = Config()
    set_seed(config.seed)
    
    logger.info("\n" + "=" * 80)
    logger.info("COMBINED DATASET GENERALIZATION EXPERIMENT")
    logger.info("=" * 80)
    logger.info(f"Binary Classification: Normal (0) vs Tumor (1)")
    logger.info(f"Strategy: Split each dataset → Combine train/val/test separately")
    logger.info(f"Train ratio: {config.train_ratio*100:.0f}%")
    logger.info(f"Val ratio: {config.val_ratio*100:.0f}%")
    logger.info(f"Test ratio: {config.test_ratio*100:.0f}%")
    
    # ========== STEP 1: LOAD AND SPLIT ALL DATASETS ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 1: LOADING AND SPLITTING DATASETS")
    logger.info("=" * 80)
    
    train_transform = get_train_transforms(config.img_size)
    val_transform = get_val_transforms(config.img_size)
    
    all_train_datasets = []
    all_val_datasets = []
    all_test_datasets = {}
    
    dataset_configs = [
        (config.dataset1_dir, "Dataset-1 (NCT-CRC-100K)"),
        (config.dataset2_dir, "Dataset-2 (CRC-VAL-7K)"),
        (config.dataset3_dir, "Dataset-3 (LC25000-Colon)"),
    ]
    
    # Alternative LC25000 paths to try
    lc25000_alternative_paths = [
        "/kaggle/input/lc25000/lung_colon_image_set/lung_colon_image_set/colon_image_sets",
        "/kaggle/input/lc25000/colon_image_sets",
        "/kaggle/input/lc25000-colon",
    ]
    
    # If dataset3 doesn't exist, try alternatives
    if not os.path.exists(config.dataset3_dir):
        logger.warning(f"⚠ LC25000 path not found: {config.dataset3_dir}")
        logger.info("Trying alternative paths...")
        for alt_path in lc25000_alternative_paths:
            if os.path.exists(alt_path):
                logger.info(f"✓ Found LC25000 at: {alt_path}")
                dataset_configs[2] = (alt_path, "Dataset-3 (LC25000-Colon)")
                break
        else:
            logger.warning("⚠ Could not find LC25000 dataset in any known location")
    
    logger.info(f"\nDataset paths to process:")
    for path, name in dataset_configs:
        exists = "✓" if os.path.exists(path) else "✗"
        logger.info(f"  {exists} {name}: {path}")
    
    for dataset_dir, dataset_name in dataset_configs:
        if not os.path.exists(dataset_dir):
            logger.warning(f"⚠ Skipping {dataset_name} - Directory not found: {dataset_dir}")
            continue
        
        logger.info(f"\n{'='*60}")
        logger.info(f"Processing {dataset_name}")
        logger.info(f"{'='*60}")
        
        # Load full dataset (already mapped to binary)
        try:
            full_dataset = ImageDataset(dataset_dir, None, dataset_name)
        except Exception as e:
            logger.error(f"❌ Error loading {dataset_name}: {e}")
            continue
        
        if len(full_dataset) == 0:
            logger.warning(f"⚠ Skipping {dataset_name} - No images found")
            continue
        
        # Split indices
        indices = np.arange(len(full_dataset))
        labels = np.array(full_dataset.labels)
        
        # Train + temp split
        train_idx, temp_idx = train_test_split(
            indices, test_size=(config.val_ratio + config.test_ratio),
            stratify=labels, random_state=config.seed
        )
        
        
        # Val + test split
        temp_labels = labels[temp_idx]
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=config.test_ratio/(config.val_ratio + config.test_ratio),
            stratify=temp_labels, random_state=config.seed
        )
        
        logger.info(f"\n{dataset_name}:")
        logger.info(f"  Total samples: {len(full_dataset)}")
        logger.info(f"  Train: {len(train_idx)} samples ({len(train_idx)/len(full_dataset)*100:.1f}%)")
        logger.info(f"  Val:   {len(val_idx)} samples ({len(val_idx)/len(full_dataset)*100:.1f}%)")
        logger.info(f"  Test:  {len(test_idx)} samples ({len(test_idx)/len(full_dataset)*100:.1f}%)")
        
        # Create split datasets
        train_ds = ImageDataset(dataset_dir, train_transform, dataset_name)
        train_ds.samples = [train_ds.samples[i] for i in train_idx]
        train_ds.labels = [train_ds.labels[i] for i in train_idx]
        all_train_datasets.append(train_ds)
        
        val_ds = ImageDataset(dataset_dir, val_transform, dataset_name)
        val_ds.samples = [val_ds.samples[i] for i in val_idx]
        val_ds.labels = [val_ds.labels[i] for i in val_idx]
        all_val_datasets.append(val_ds)
        
        test_ds = ImageDataset(dataset_dir, val_transform, dataset_name)
        test_ds.samples = [test_ds.samples[i] for i in test_idx]
        test_ds.labels = [test_ds.labels[i] for i in test_idx]
        test_dataset_key = f"Test-{dataset_name}"
        all_test_datasets[test_dataset_key] = test_ds
        logger.info(f"  ✓ Test dataset created: {test_dataset_key} with {len(test_ds)} samples")
    
    if not all_train_datasets:
        logger.error("❌ No datasets loaded!")
        return
    
    logger.info(f"\n{'='*80}")
    logger.info(f"DATASETS LOADED SUCCESSFULLY")
    logger.info(f"{'='*80}")
    logger.info(f"Number of datasets loaded: {len(all_train_datasets)}")
    logger.info(f"Test datasets created: {list(all_test_datasets.keys())}")
    
    # Binary classification: 2 classes
    num_classes = 2
    class_names = ['Normal', 'Tumor']
    logger.info(f"\n✓ Binary Classification: {num_classes} classes")
    logger.info(f"  Classes: {class_names}")
    
    # ========== STEP 2: COMBINE DATASETS ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 2: COMBINING DATASETS")
    logger.info("=" * 80)
    
    combined_train = ConcatDataset(all_train_datasets)
    combined_val = ConcatDataset(all_val_datasets)
    
    logger.info(f"✓ Combined Training Set: {len(combined_train)} samples")
    logger.info(f"✓ Combined Validation Set: {len(combined_val)} samples")
    
    # Create combined test dataset
    combined_test_samples = []
    combined_test_labels = []
    for test_ds in all_test_datasets.values():
        combined_test_samples.extend(test_ds.samples)
        combined_test_labels.extend(test_ds.labels)
    
    combined_test_ds = ImageDataset(config.dataset1_dir, val_transform, "Combined-Test")
    combined_test_ds.samples = combined_test_samples
    combined_test_ds.labels = combined_test_labels
    combined_test_ds.class_names = class_names
    all_test_datasets["Test-Combined-All"] = combined_test_ds
    
    logger.info(f"✓ Combined Test Set: {len(combined_test_ds)} samples")
    
    # ========== STEP 3: CREATE DATALOADERS ==========
    train_loader = DataLoader(combined_train, batch_size=config.batch_size, shuffle=True,
                              num_workers=config.num_workers, pin_memory=True)
    val_loader = DataLoader(combined_val, batch_size=config.batch_size, shuffle=False,
                           num_workers=config.num_workers, pin_memory=True)
    
    # ========== STEP 4: TRAIN MODEL ==========
    model = ColonCancerClassifier(num_classes=num_classes, pretrained=True)
    trainer = Trainer(model, train_loader, val_loader, config)
    trainer.train()
    trainer.load_best_model()
    
    # ========== STEP 5: EVALUATE ON ALL TEST SPLITS ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 5: EVALUATING ON TEST SPLITS")
    logger.info("=" * 80)
    logger.info(f"Total test datasets to evaluate: {len(all_test_datasets)}")
    for test_name in all_test_datasets.keys():
        logger.info(f"  - {test_name}")
    
    evaluator = Evaluator(trainer.model, config)
    
    for test_name, test_ds in all_test_datasets.items():
        test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False,
                                num_workers=config.num_workers, pin_memory=True)
        evaluator.evaluate_dataset(test_loader, test_name, test_ds.class_names)
    
    # ========== STEP 6: VISUALIZE AND COMPARE ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 6: RESULTS SUMMARY")
    logger.info("=" * 80)
    
    # Print detailed summary
    logger.info("\n📊 DETAILED RESULTS FOR EACH TEST SPLIT:")
    for test_name, res in evaluator.results.items():
        logger.info(f"\n{test_name}:")
        logger.info(f"  Samples: {res['num_samples']}")
        logger.info(f"  Classes: {res['class_names']}")
        logger.info(f"  Accuracy: {res['accuracy']*100:.2f}%")
        logger.info(f"  Balanced Accuracy: {res['balanced_accuracy']*100:.2f}%")
        logger.info(f"  Precision: {res['precision']:.4f}")
        logger.info(f"  Recall: {res['recall']:.4f}")
        logger.info(f"  F1-Score: {res['f1_score']:.4f}")
        logger.info(f"  ROC-AUC: {res['roc_auc']:.4f}")
    
    evaluator.compare_results()
    evaluator.plot_confusion_matrices()
    evaluator.plot_roc_curves()
    
    # Plot class distribution
    plot_class_distribution(evaluator.results, config)
    
    # Save results
    results_json = {}
    for name, res in evaluator.results.items():
        results_json[name] = {
            'dataset': res['dataset'],
            'num_samples': res['num_samples'],
            'accuracy': res['accuracy'],
            'f1_score': res['f1_score'],
            'roc_auc': res['roc_auc'],
        }
    
    output_path = os.path.join(config.output_dir, 'combined_generalization_results.json')
    with open(output_path, 'w') as f:
        json.dump(results_json, f, indent=2)
    logger.info(f"✓ Results saved: {output_path}")
    
    logger.info("\n" + "=" * 80)
    logger.info("✓ COMBINED GENERALIZATION EXPERIMENT COMPLETE!")
    logger.info("=" * 80)


if __name__ == "__main__":
    main()


2026-04-03 14:51:40,583 - INFO - 
2026-04-03 14:51:40,584 - INFO - COMBINED DATASET GENERALIZATION EXPERIMENT
2026-04-03 14:51:40,585 - INFO - ================================================================================
2026-04-03 14:51:40,586 - INFO - Binary Classification: Normal (0) vs Tumor (1)
2026-04-03 14:51:40,586 - INFO - Strategy: Split each dataset → Combine train/val/test separately
2026-04-03 14:51:40,587 - INFO - Train ratio: 70%
2026-04-03 14:51:40,588 - INFO - Val ratio: 15%
2026-04-03 14:51:40,590 - INFO - Test ratio: 15%
2026-04-03 14:51:40,590 - INFO - 
2026-04-03 14:51:40,591 - INFO - STEP 1: LOADING AND SPLITTING DATASETS
2026-04-03 14:51:40,592 - INFO - ================================================================================
2026-04-03 14:51:40,597 - INFO - 
Dataset paths to process:
2026-04-03 14:51:40,600 - INFO -   ✓ Dataset-1 (NCT-CRC-100K): /kaggle/input/datasets/imrankhan77/nct-crc-he-100k/NCT-CRC-HE-100K
2026-04-03 14:51:40,605 - INFO -   ✓ Data

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 183MB/s] 
/tmp/ipykernel_55/1706208945.py:238: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=config.use_amp)
2026-04-03 14:51:48,254 - INFO - 
2026-04-03 14:51:48,255 - INFO - TRAINING ON COMBINED DATASETS
2026-04-03 14:51:48,255 - INFO - ================================================================================
Epoch 1/10 [Train]:   0%|          | 0/1282 [00:00<?, ?it/s]/tmp/ipykernel_55/1706208945.py:252: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.config.use_amp):
[Validation]:   0%|          | 0/275 [00:00<?, ?it/s]/tmp/ipykernel_55/1706208945.py:280: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.config.use_am

In [3]:
import shutil
import os
from IPython.display import FileLink

# 1. Path to your outputs
output_path = "/kaggle/working/combined_generalization_outputs"
zip_name = 'colon_cancer_robust_results'

# 2. Create the zip file
# This takes everything in /outputs and puts it into colon_cancer_results.zip
shutil.make_archive(zip_name, 'zip', output_path)

print(f"✅ Created {zip_name}.zip")

# 3. Generate the download link
FileLink(f'{zip_name}.zip')

✅ Created colon_cancer_robust_results.zip


/kaggle/working/colon_cancer_robust_results.zip